# FLUX Chat Server — Kaggle & Colab Compatible

**Goal:** Run the FLUX Flask Chat application in a notebook environment with GPU access.

### What this notebook does

1. Detect environment (Kaggle vs Colab) + setup persistent storage
2. Clone/pull FLUX repository
3. Install dependencies
4. Initialize logger, detect hardware, load HF_TOKEN
5. Download Flux-beta.flx checkpoint (or fallback to phase8.phase.pt)
6. Start Flask server with ngrok tunnel for external access
7. Interactive cells for testing endpoints

### Secrets Required

- `HF_TOKEN`: HuggingFace token for downloading checkpoints
- `NGROK_AUTH_TOKEN` (optional): For persistent ngrok URLs

### Setting up Secrets

**Kaggle:** Add-ons → Secrets → Add `HF_TOKEN`  
**Colab:** 🔑 icon in left sidebar → Add `HF_TOKEN`

---
## Cell 1: Detect Environment & Setup Storage

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# ─────────────────────────────────────────────
# 0. Detect Environment: Kaggle vs Colab vs Local
# ─────────────────────────────────────────────
IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
IS_LOCAL = not IS_KAGGLE and not IS_COLAB

print(f"Environment Detection:")
print(f"  Kaggle: {IS_KAGGLE}")
print(f"  Colab: {IS_COLAB}")
print(f"  Local: {IS_LOCAL}")

REPO_URL = "https://github.com/Unseengap/FLUX.git"

# ─────────────────────────────────────────────
# 1. Setup directories based on environment
# ─────────────────────────────────────────────
if IS_KAGGLE:
    WORK_DIR = Path('/kaggle/working/FLUX')
    PERSISTENT_DIR = Path('/kaggle/working')
    print(f"  ✓ Kaggle environment — outputs to /kaggle/working")
    
elif IS_COLAB:
    # Mount Google Drive for persistent storage
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    
    DRIVE_FLUX = Path('/content/drive/MyDrive/FLUX')
    DRIVE_FLUX.mkdir(parents=True, exist_ok=True)
    DRIVE_CHECKPOINTS = DRIVE_FLUX / 'checkpoints'
    DRIVE_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
    DRIVE_LOGS = DRIVE_FLUX / 'logs'
    DRIVE_LOGS.mkdir(parents=True, exist_ok=True)
    
    WORK_DIR = Path('/content/FLUX')
    PERSISTENT_DIR = DRIVE_FLUX
    print(f"  ✓ Google Drive mounted — persistent storage at {DRIVE_FLUX}")
    
else:
    WORK_DIR = Path.cwd()
    PERSISTENT_DIR = WORK_DIR
    print(f"  ✓ Local environment — using {WORK_DIR}")

print(f"\n  Work directory: {WORK_DIR}")
print(f"  Persistent storage: {PERSISTENT_DIR}")

---
## Cell 2: Clone / Pull Repository

In [ ]:
import subprocess
import shutil

if IS_LOCAL:
    print("  ℹ Local environment — assuming repo already cloned")
    if not WORK_DIR.exists():
        raise FileNotFoundError(f"Expected FLUX repo at {WORK_DIR}")
else:
    # Clone or pull
    if WORK_DIR.exists() and (WORK_DIR / '.git').exists():
        print("ℹ Repo exists — pulling latest changes...")
        subprocess.run(['git', 'remote', 'set-url', 'origin', REPO_URL],
                       cwd=str(WORK_DIR), capture_output=True)
        subprocess.run(['git', 'checkout', '.'], cwd=str(WORK_DIR), capture_output=True)
        subprocess.run(['git', 'clean', '-fd'], cwd=str(WORK_DIR), capture_output=True)
        subprocess.run(['git', 'fetch', '--all'], cwd=str(WORK_DIR), capture_output=True)
        result = subprocess.run(['git', 'reset', '--hard', 'origin/main'],
                                cwd=str(WORK_DIR), capture_output=True, text=True)
        print(result.stdout.strip() or result.stderr.strip())
        
        head = subprocess.run(['git', 'log', '--oneline', '-3'],
                              cwd=str(WORK_DIR), capture_output=True, text=True)
        print(f"\n  Latest commits:\n{head.stdout.strip()}")
        print("\n✓ Pulled & reset to origin/main")
    else:
        if WORK_DIR.exists():
            shutil.rmtree(str(WORK_DIR))
        print(f"ℹ Cloning {REPO_URL}...")
        subprocess.run(['git', 'clone', REPO_URL, str(WORK_DIR)], check=True)
        print("✓ Cloned successfully")

os.chdir(str(WORK_DIR))
print(f"\nWorking directory: {os.getcwd()}")

# Symlink checkpoints/logs to persistent storage (Colab only)
if IS_COLAB:
    local_ckpt = WORK_DIR / 'checkpoints'
    if local_ckpt.is_symlink():
        local_ckpt.unlink()
    elif local_ckpt.exists():
        for f in local_ckpt.glob('*.pt'):
            dst = DRIVE_CHECKPOINTS / f.name
            if not dst.exists():
                shutil.copy2(str(f), str(dst))
        for f in local_ckpt.glob('*.flx'):
            dst = DRIVE_CHECKPOINTS / f.name
            if not dst.exists():
                shutil.copy2(str(f), str(dst))
        shutil.rmtree(str(local_ckpt))
    local_ckpt.symlink_to(DRIVE_CHECKPOINTS)
    print(f"  ✓ checkpoints/ → {DRIVE_CHECKPOINTS}")
    
    local_logs = WORK_DIR / 'logs'
    if local_logs.is_symlink():
        local_logs.unlink()
    elif local_logs.exists():
        for f in local_logs.glob('*.log'):
            dst = DRIVE_LOGS / f.name
            if not dst.exists():
                shutil.copy2(str(f), str(dst))
        shutil.rmtree(str(local_logs))
    local_logs.symlink_to(DRIVE_LOGS)
    print(f"  ✓ logs/ → {DRIVE_LOGS}")

# Run setup.py
subprocess.run(['python', 'setup.py'], cwd=str(WORK_DIR), capture_output=True)
print("✓ setup.py executed")

---
## Cell 3: Install Dependencies

In [ ]:
# Core dependencies
!pip install -q flask pyngrok datasets rich faiss-cpu huggingface_hub matplotlib transformers scipy

print("\n✓ Dependencies installed")

---
## Cell 4: Initialize Logger, Detect Hardware & Load Secrets

In [ ]:
import sys
import torch
from pathlib import Path

# ─────────────────────────────────────────────
# Add project paths
# ─────────────────────────────────────────────
sys.path.insert(0, str(Path.cwd()))
for phase in ['phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase6', 'phase7', 'phase8']:
    sys.path.insert(0, str(Path.cwd() / 'phases' / phase))

from flux_utils import (
    get_device, get_hardware_info, PhaseLogger, _resolve_hf_token,
    CHECKPOINT_DIR, HF_REPO_ID,
)

# ─────────────────────────────────────────────
# Initialize logger
# ─────────────────────────────────────────────
log = PhaseLogger(phase=100)  # Use phase 100 for app server
log.separator("FLUX Chat Server — Notebook Environment")
log.cell_start("Cell 4 — Hardware & Secrets")

# ─────────────────────────────────────────────
# Detect hardware
# ─────────────────────────────────────────────
DEVICE = get_device()
hw = get_hardware_info()
log.info(f"Device: {DEVICE}")
log.info(f"PyTorch: {torch.__version__}")

for k, v in hw.items():
    print(f"  {k}: {v}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    log.info(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    print(f"  ✓ GPU acceleration available!")
else:
    print(f"  ⚠ No GPU — will use CPU (slower)")

# ─────────────────────────────────────────────
# Load HuggingFace token
# ─────────────────────────────────────────────
HF_TOKEN = None

# Try Kaggle secrets
if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
        if HF_TOKEN:
            HF_TOKEN = HF_TOKEN.strip()
            print(f"  ✓ HF_TOKEN loaded from Kaggle Secrets")
    except Exception:
        pass

# Try Colab secrets
if IS_COLAB and not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
        if HF_TOKEN:
            HF_TOKEN = HF_TOKEN.strip()
            print(f"  ✓ HF_TOKEN loaded from Colab Secrets")
    except Exception:
        pass

# Fallback to env/file
if not HF_TOKEN:
    HF_TOKEN = _resolve_hf_token()

if HF_TOKEN:
    log.success("HuggingFace token loaded")
else:
    log.warning("No HuggingFace token — may not be able to download checkpoint")
    print("  ⚠ Add HF_TOKEN to your environment secrets")

# ─────────────────────────────────────────────
# Load ngrok token (optional)
# ─────────────────────────────────────────────
NGROK_TOKEN = None

if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        NGROK_TOKEN = UserSecretsClient().get_secret('NGROK_AUTH_TOKEN')
    except Exception:
        pass

if IS_COLAB and not NGROK_TOKEN:
    try:
        from google.colab import userdata
        NGROK_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
    except Exception:
        pass

if NGROK_TOKEN:
    print(f"  ✓ NGROK_AUTH_TOKEN loaded (persistent URLs available)")
else:
    print(f"  ℹ No NGROK_AUTH_TOKEN — will use temporary URLs")

log.cell_end("Cell 4 — Hardware & Secrets", "PASS")

---
## Cell 5: Download Flux-beta.flx Checkpoint

In [ ]:
log.cell_start("Cell 5 — Download Checkpoint")

from huggingface_hub import hf_hub_download

# Check for existing Flux-beta.flx
flx_path = CHECKPOINT_DIR / 'Flux-beta.flx'
phase8_path = CHECKPOINT_DIR / 'phase8.phase.pt'

if flx_path.exists():
    size_mb = flx_path.stat().st_size / 1e6
    log.info(f"Flux-beta.flx already exists ({size_mb:.1f} MB)")
    MODEL_PATH = str(flx_path)
else:
    # Try downloading Flux-beta.flx
    log.info("Downloading Flux-beta.flx from HuggingFace Hub...")
    try:
        hf_hub_download(
            repo_id=HF_REPO_ID,
            filename="checkpoints/Flux-beta.flx",
            local_dir=str(CHECKPOINT_DIR.parent),
            token=HF_TOKEN,
        )
        size_mb = flx_path.stat().st_size / 1e6
        log.success(f"Flux-beta.flx downloaded ({size_mb:.1f} MB)")
        MODEL_PATH = str(flx_path)
    except Exception as e:
        log.warning(f"Could not download Flux-beta.flx: {e}")
        
        # Fallback to phase8.phase.pt
        if phase8_path.exists():
            size_mb = phase8_path.stat().st_size / 1e6
            log.info(f"Using existing phase8.phase.pt ({size_mb:.1f} MB)")
            MODEL_PATH = str(phase8_path)
        else:
            log.info("Downloading phase8.phase.pt as fallback...")
            try:
                hf_hub_download(
                    repo_id=HF_REPO_ID,
                    filename="checkpoints/phase8.phase.pt",
                    local_dir=str(CHECKPOINT_DIR.parent),
                    token=HF_TOKEN,
                )
                size_mb = phase8_path.stat().st_size / 1e6
                log.success(f"phase8.phase.pt downloaded ({size_mb:.1f} MB)")
                MODEL_PATH = str(phase8_path)
            except Exception as e2:
                log.error(f"Could not download any checkpoint: {e2}")
                raise FileNotFoundError("No checkpoint available")

print(f"\n  Model path: {MODEL_PATH}")
log.cell_end("Cell 5 — Download Checkpoint", "PASS")

---
## Cell 6: Start Flask Server with ngrok Tunnel

This cell:
1. Creates the Flask app
2. Loads the FLUX model
3. Starts an ngrok tunnel for external access
4. Runs the server in a background thread

**The public URL will be displayed below — use it to access the chat interface!**

In [ ]:
log.cell_start("Cell 6 — Start Flask Server")

import threading
import time
from pyngrok import ngrok, conf

# ─────────────────────────────────────────────
# Configure ngrok
# ─────────────────────────────────────────────
if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN
    print("  ✓ ngrok authenticated")

# ─────────────────────────────────────────────
# Create Flask app with custom config
# ─────────────────────────────────────────────
sys.path.insert(0, str(WORK_DIR))
sys.path.insert(0, str(WORK_DIR / 'flux_app'))

from flux_app import create_app

app = create_app()
app.config['MODEL_PATH'] = MODEL_PATH
app.config['DEVICE'] = DEVICE
app.config['LOG_LEVEL'] = 'INFO'  # Less verbose for notebook

print(f"  ✓ Flask app created")
print(f"    Model: {MODEL_PATH}")
print(f"    Device: {DEVICE}")

# ─────────────────────────────────────────────
# Pre-load the model
# ─────────────────────────────────────────────
print("\n  Loading FLUX model (this may take 30-60 seconds)...")
load_start = time.time()

with app.app_context():
    try:
        stats = app.flux_service.load_model()
        load_time = time.time() - load_start
        print(f"  ✓ Model loaded in {load_time:.1f}s")
        print(f"    Parameters: {stats.get('total_params', 'N/A')}")
        print(f"    Episodic entries: {stats.get('episodic_entries', 0)}")
        print(f"    Device: {stats.get('device', DEVICE)}")
    except Exception as e:
        print(f"  ⚠ Model loading error: {e}")
        print(f"    The server will start, but you'll need to load the model via /model/load")

# ─────────────────────────────────────────────
# Start ngrok tunnel
# ─────────────────────────────────────────────
PORT = 5000

# Kill any existing ngrok tunnels
try:
    ngrok.kill()
except:
    pass

public_url = ngrok.connect(PORT, "http")
print(f"\n" + "═" * 60)
print(f"  🌐 PUBLIC URL: {public_url}")
print(f"═" * 60)
print(f"\n  Open this URL in your browser to access the FLUX Chat interface!")
print(f"  (The URL will remain active as long as this notebook is running)")

# ─────────────────────────────────────────────
# Run Flask in background thread
# ─────────────────────────────────────────────
def run_flask():
    """Run Flask server in background."""
    app.run(
        host='0.0.0.0',
        port=PORT,
        debug=False,  # No debug in notebook (conflicts with reloader)
        use_reloader=False,
        threaded=True,
    )

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

print(f"\n  ✓ Flask server running on port {PORT}")
print(f"  ✓ Server thread started (daemon mode)")

# Wait a moment for server to start
time.sleep(2)

log.success(f"Server started at {public_url}")
log.cell_end("Cell 6 — Start Flask Server", "PASS")

---
## Cell 7: Test API Endpoints

Run these cells to test the API directly from the notebook.

In [ ]:
# Test health endpoint
import requests

response = requests.get(f"http://localhost:{PORT}/api/health")
print("Health Check:")
print(f"  Status: {response.status_code}")
print(f"  Response: {response.json()}")

In [ ]:
# Test model stats
response = requests.get(f"http://localhost:{PORT}/api/stats")
print("Model Stats:")
print(f"  Status: {response.status_code}")
data = response.json()
if data.get('loaded'):
    print(f"  Model loaded: ✓")
    for k, v in data.get('stats', {}).items():
        print(f"    {k}: {v}")
else:
    print(f"  Model loaded: ✗")

In [ ]:
# Test generation (synchronous for notebook)
import json

prompt = "The physics of resonance fields explains"

print(f"Testing generation with prompt: '{prompt}'")
print("-" * 50)

response = requests.post(
    f"http://localhost:{PORT}/api/generate",
    json={
        'prompt': prompt,
        'max_length': 50,
        'temperature': 0.8,
    },
    stream=True,
)

generated_text = ""
for line in response.iter_lines():
    if line:
        line = line.decode('utf-8')
        if line.startswith('data: '):
            try:
                event = json.loads(line[6:])
                if event.get('type') == 'char':
                    char = event.get('char', '')
                    generated_text += char
                    print(char, end='', flush=True)
                elif event.get('type') == 'phase':
                    phase = event.get('phase', '?')
                    status = event.get('status', '')
                    if status == 'complete':
                        pass  # Silent for cleaner output
            except json.JSONDecodeError:
                pass

print("\n" + "-" * 50)
print(f"Generated {len(generated_text)} characters")

In [ ]:
# Test one-shot learning
fact = "The FLUX architecture was created in 2026 as a physics-inspired alternative to transformers."

print(f"Teaching fact: '{fact}'")
response = requests.post(
    f"http://localhost:{PORT}/api/learn",
    json={'fact': fact}
)

print(f"\nResponse:")
print(f"  Status: {response.status_code}")
for k, v in response.json().items():
    print(f"    {k}: {v}")

In [ ]:
# Test recall
query = "When was FLUX created?"

print(f"Recall query: '{query}'")
response = requests.post(
    f"http://localhost:{PORT}/api/recall",
    json={'query': query, 'k': 3}
)

print(f"\nResults:")
data = response.json()
for i, result in enumerate(data.get('results', [])):
    print(f"  {i+1}. (sim={result.get('similarity', 0):.4f}) {result.get('fact', '')[:80]}...")

---
## Cell 8: View Server Logs

Check the Flask app logs for debugging.

In [ ]:
# Display recent Flask logs
log_file = WORK_DIR / 'logs' / 'flask_app.log'

if log_file.exists():
    print("=" * 60)
    print("Recent Flask App Logs (last 50 lines):")
    print("=" * 60)
    with open(log_file, 'r') as f:
        lines = f.readlines()
        for line in lines[-50:]:
            print(line.rstrip())
else:
    print("No log file found yet.")

---
## Cell 9: Interactive Chat (Optional)

Simple text-based chat interface directly in the notebook.

In [ ]:
# Simple interactive chat
def chat(prompt: str, max_length: int = 100, temperature: float = 0.8) -> str:
    """Send a prompt and get a response."""
    response = requests.post(
        f"http://localhost:{PORT}/api/generate",
        json={
            'prompt': prompt,
            'max_length': max_length,
            'temperature': temperature,
        },
        stream=True,
    )
    
    text = ""
    for line in response.iter_lines():
        if line:
            line = line.decode('utf-8')
            if line.startswith('data: '):
                try:
                    event = json.loads(line[6:])
                    if event.get('type') == 'char':
                        text += event.get('char', '')
                except:
                    pass
    return text

# Example usage:
print("Chat function loaded! Use: chat('your prompt here')")
print("\nExample:")
response = chat("Hello, I am FLUX. I am")
print(f"FLUX: {response}")

---
## Cell 10: Cleanup (Optional)

Run this cell to stop the server and cleanup resources.

In [ ]:
# Stop ngrok tunnel
try:
    ngrok.kill()
    print("✓ ngrok tunnel closed")
except:
    pass

# Note: Flask thread is daemon, so it will stop when notebook stops
print("\nTo fully stop the server, restart the notebook kernel.")
print("The Flask thread is running in daemon mode and will stop automatically.")

---
## Troubleshooting

### Common Issues:

1. **Model not loading**: Check that `Flux-beta.flx` or `phase8.phase.pt` exists in `checkpoints/`
2. **ngrok connection failed**: Your session may have expired. Run Cell 6 again.
3. **Out of memory**: Try reducing `max_length` in generation calls
4. **Slow generation**: Ensure you're using GPU (check Device in Cell 4)

### Manual Model Loading:

If the model didn't load automatically, use this:
```python
requests.post(f"http://localhost:{PORT}/model/load")
```